In [ ]:
# 1. Initialization, GCP AUTH, & Metadata Fetch
from google.colab import auth
from google.cloud import bigquery
from pandas_gbq import to_gbq
import pandas as pd
import requests
import pytz
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

# Proses Autentikasi Google Cloud
auth.authenticate_user()

project_id = 'fsb-12345'
dataset_id_slv = 'slv_bike_store'
dataset_id_brz = 'brz_bike_store'

client = bigquery.Client(project=project_id)

# Mengambil katalog Metadata dari GitHub
url_metadata = "https://raw.githubusercontent.com/bachtiyarma/FSB/main/Project%20-%20BikeStore/ListDataBikeStore.json"
response = requests.get(url_metadata)

if response.status_code == 200:
    list_sheet_id = response.json()
    print("✅ Variabel 'list_sheet_id' berhasil didefinisikan ulang dari GitHub.")
else:
    raise Exception(f"Gagal mengambil metadata JSON dari GitHub. Status: {response.status_code}")

# Membuat Dataset Silver
dataset_ref = bigquery.Dataset(f'{project_id}.{dataset_id_slv}')
dataset_ref.location = "US"
dataset_slv = client.create_dataset(dataset_ref, exists_ok=True)
print(f"✅ Dataset '{dataset_id_slv}' siap digunakan untuk Trusted Data.")

✅ Variabel 'list_sheet_id' berhasil didefinisikan ulang dari GitHub.
✅ Dataset 'slv_bike_store' siap digunakan untuk Trusted Data.


In [ ]:
# 2. Helper Functions
def extract_data_from_gbq(query):
    return client.query(query).to_dataframe()

def load_data_to_gbq(data, table_name, id_dataset):
    destination = f'{id_dataset}.{table_name}'
    for col in data.columns:
        if data[col].dtype == 'object':
            data[col] = data[col].astype(str)

    to_gbq(
        data,
        destination_table=destination,
        project_id=project_id,
        if_exists='replace',
        progress_bar=False
    )
    print(f"   [SUCCESS] Dimuat ke BigQuery: {destination}")

def add_metadata_silver(data):
    now_utc = datetime.now(timezone.utc)
    date_now = now_utc.astimezone(pytz.timezone('Asia/Jakarta'))

    data_add_md = data.copy()
    data_add_md['job_updated_date'] = pd.to_datetime(date_now)
    data_add_md['job_updated_by'] = 'SYSTEM_SILVER'
    return data_add_md

In [ ]:
# 3. Run Automated ETL Pipeline
print("\n=== Memulai Proses Pembersihan & Standardisasi (Silver Layer) ===")

for key, value in list_sheet_id.items():
    table_name = key

    # Ekstraksi dari tabel mentah di Bronze Layer
    query_extract = f"SELECT * FROM {project_id}.{dataset_id_brz}.raw_{table_name}"
    print(f'\n⚙️ Mengolah data: raw_{table_name} -> {table_name}')

    try:
        data_bronze = extract_data_from_gbq(query_extract)

        # Mengambil key_name dari katalog metadata
        key_name = str(value['key_name']).lower().strip()

        # Pembersihan Duplikasi & Missing Values pada Primary Key
        if key_name in data_bronze.columns:
            data_bronze = data_bronze.dropna(subset=[key_name])
            data_bronze = data_bronze.drop_duplicates(subset=[key_name])
        else:
            print(f"   [Warning] Key '{key_name}' tidak ditemukan di tabel. Melewati langkah deduplikasi.")

        # Standardisasi Kolom Teks
        col_object = data_bronze.select_dtypes(include='object').columns
        for col in col_object:
            if col in ['job_created_by', 'job_created_date']:
                continue
            data_bronze[col] = data_bronze[col].astype(str).str.strip().str.upper()
            data_bronze[col] = data_bronze[col].replace({'NAN': None, 'NONE': None, '<NA>': None})

        # Standardisasi Kolom Tanggal
        for col in data_bronze.columns:
            if '_date' in col:
                data_bronze[col] = pd.to_datetime(data_bronze[col], errors='coerce')

        # Standarisasi Tipe Data Numerik
        numeric_cols = ['quantity', 'list_price', 'discount', 'model_year', 'active']
        for col in numeric_cols:
            if col in data_bronze.columns:
                data_bronze[col] = pd.to_numeric(data_bronze[col], errors='coerce')

        # Nambahkan Metadata Silver
        data_silver_md = add_metadata_silver(data_bronze)

        # Nampilkan sampel data hasil pembersihan
        display(data_silver_md.head(2))

        # Load ke Dataset Silver
        load_data_to_gbq(
            data=data_silver_md,
            table_name=table_name,
            id_dataset=dataset_id_slv
        )

    except Exception as e:
        print(f"❌ Gagal memproses Silver Layer untuk tabel {table_name}. Error: {e}")

print("\n=== Pipeline Silver Layer Selesai Dijalankan ===")


=== Memulai Proses Pembersihan & Standardisasi (Silver Layer) ===

⚙️ Mengolah data: raw_brands -> brands


,brand_id,brand_name,job_created_date,job_created_by,job_updated_date,job_updated_by
0,BRAND0001,ELECTRA,2026-06-12 13:55:15.092956+00:00,SYSTEM,2026-06-12 21:13:21.632764+07:00,SYSTEM_SILVER
1,BRAND0002,HARO,2026-06-12 13:55:15.092956+00:00,SYSTEM,2026-06-12 21:13:21.632764+07:00,SYSTEM_SILVER


   [SUCCESS] Dimuat ke BigQuery: slv_bike_store.brands

⚙️ Mengolah data: raw_categories -> categories


,category_id,category_name,job_created_date,job_created_by,job_updated_date,job_updated_by
0,CATEGORY0001,CHILDREN BICYCLES,2026-06-12 13:55:21.986889+00:00,SYSTEM,2026-06-12 21:13:28.293348+07:00,SYSTEM_SILVER
1,CATEGORY0002,COMFORT BICYCLES,2026-06-12 13:55:21.986889+00:00,SYSTEM,2026-06-12 21:13:28.293348+07:00,SYSTEM_SILVER


   [SUCCESS] Dimuat ke BigQuery: slv_bike_store.categories

⚙️ Mengolah data: raw_customers -> customers


...

   [SUCCESS] Dimuat ke BigQuery: slv_bike_store.staffs

⚙️ Mengolah data: raw_stocks -> stocks


,stock_id,store_id,product_id,quantity,job_created_date,job_created_by,job_updated_date,job_updated_by
0,STOCK0006,STORE0001,PRODUCT0006,0,2026-06-12 13:55:52.393009+00:00,SYSTEM,2026-06-12 21:14:08.660511+07:00,SYSTEM_SILVER
1,STOCK0008,STORE0001,PRODUCT0008,0,2026-06-12 13:55:52.393009+00:00,SYSTEM,2026-06-12 21:14:08.660511+07:00,SYSTEM_SILVER


   [SUCCESS] Dimuat ke BigQuery: slv_bike_store.stocks

⚙️ Mengolah data: raw_stores -> stores


,store_id,store_name,phone,email,street,city,state,zip_code,job_created_date,job_created_by,job_updated_date,job_updated_by
0,STORE0001,SANTA CRUZ BIKES,(831); 476-4321,SANTACRUZ@BIKES.SHOP,3700 PORTOLA DRIVE,SANTA CRUZ,CA,95060,2026-06-12 13:55:56.410469+00:00,SYSTEM,2026-06-12 21:14:14.041507+07:00,SYSTEM_SILVER
1,STORE0002,BALDWIN BIKES,(516); 379-8888,BALDWIN@BIKES.SHOP,4200 CHESTNUT LANE,BALDWIN,NY,11432,2026-06-12 13:55:56.410469+00:00,SYSTEM,2026-06-12 21:14:14.041507+07:00,SYSTEM_SILVER


   [SUCCESS] Dimuat ke BigQuery: slv_bike_store.stores

=== Pipeline Silver Layer Selesai Dijalankan ===
